# Sta-RU Video Dubbing

Doblaje de videos RU → EN (u otros idiomas) con clonación de voz, usando los subtítulos pre-traducidos del repo.

**Pipeline:**
1. Descarga el video (YouTube con `yt-dlp` o copia desde Drive)
2. Extrae un sample de voz limpio del audio original (~10s)
3. Genera audio por segmento con XTTS-v2 clonando la voz
4. Ajusta velocidad de cada segmento para encajar en el timing original (rubberband, sin alterar pitch)
5. Reemplaza la pista de audio en el video y exporta

**Requisitos:** Runtime con GPU (Runtime → Change runtime type → T4 GPU).

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

Tarda ~3-5 min la primera vez.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg rubberband-cli

In [ ]:
!pip install -q yt-dlp coqui-tts pyrubberband srt soundfile numpy

## 3. Clonar el repo con los subtítulos traducidos

In [ ]:
import os
if not os.path.exists('/content/Sta-RU'):
    !git clone https://github.com/lazy-money/sta-ru.git /content/Sta-RU
%cd /content/Sta-RU

## 4. (Opcional) Montar Google Drive

Descomentá si los videos están en Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 5. Configuración

Editá las variables abajo para procesar un video.

In [ ]:
from pathlib import Path

# --- ENTRADA ---
# Opción A: URL de YouTube
VIDEO_SOURCE = "https://www.youtube.com/watch?v=XXXXXXXXXXX"
# Opción B: ruta local o de Drive (descomentá y dejá la A vacía)
# VIDEO_SOURCE = "/content/drive/MyDrive/videos-ru/file.mp4"

# Subtítulo en inglés (el del repo)
SRT_EN = "Subtitles-EN/2016-07-12 - 1 - Standing wave in the inductor (early 2016)-EN.txt"

# --- SALIDA ---
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- TTS ---
TARGET_LANG = "en"  # en, es, fr, de, it, pt, pl, tr, ru, nl, cs, ar, zh-cn, ja, hu, ko

# --- AJUSTE DE TIEMPO ---
# Velocidad máxima permitida (1.4 = hasta 40% más rápido). Más allá suena artificial.
MAX_SPEED_RATIO = 1.4
# Si la traducción es más corta que el segmento original, dejamos silencio (no estiramos)

# --- VOZ DE REFERENCIA ---
# Rango (en segundos) del audio original para usar como sample de voz.
# None = auto: usa el segmento más largo entre 6-15s.
VOICE_REF_START = None
VOICE_REF_END = None

print(f"Video source: {VIDEO_SOURCE}")
print(f"SRT:          {SRT_EN}")
print(f"Idioma:       {TARGET_LANG}")
print(f"Salida:       {OUTPUT_DIR}")

## 6. Descargar/copiar el video y extraer audio

In [ ]:
import subprocess

VIDEO_PATH = str(OUTPUT_DIR / "input_video.mp4")
ORIG_AUDIO = str(OUTPUT_DIR / "orig_audio.wav")

if VIDEO_SOURCE.startswith("http"):
    subprocess.run([
        "yt-dlp", "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "--merge-output-format", "mp4",
        "-o", VIDEO_PATH, VIDEO_SOURCE
    ], check=True)
else:
    subprocess.run(["cp", VIDEO_SOURCE, VIDEO_PATH], check=True)

# Extraer audio mono 24kHz (sample rate de XTTS)
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", VIDEO_PATH, "-ac", "1", "-ar", "24000", ORIG_AUDIO
], check=True)

print(f"Video: {VIDEO_PATH}")
print(f"Audio: {ORIG_AUDIO}")

## 7. Parsear el SRT inglés

In [ ]:
import srt

with open(SRT_EN, "r", encoding="utf-8") as f:
    subs = list(srt.parse(f.read()))

print(f"Segmentos cargados: {len(subs)}")
print(f"Primero: [{subs[0].start} → {subs[0].end}] {subs[0].content}")
print(f"Último:  [{subs[-1].start} → {subs[-1].end}] {subs[-1].content}")

## 8. Extraer sample de voz para clonación

In [ ]:
import soundfile as sf

orig, sr = sf.read(ORIG_AUDIO)

if VOICE_REF_START is None or VOICE_REF_END is None:
    # Auto: el segmento más largo entre 6 y 15 segundos
    candidates = [
        s for s in subs
        if 6 <= (s.end - s.start).total_seconds() <= 15
    ]
    if candidates:
        ref = max(candidates, key=lambda s: (s.end - s.start).total_seconds())
        start_s = ref.start.total_seconds()
        end_s = ref.end.total_seconds()
    else:
        # Fallback: primeros 10s después del segundo 1
        start_s, end_s = 1.0, 11.0
else:
    start_s, end_s = VOICE_REF_START, VOICE_REF_END

ref_audio = orig[int(start_s * sr):int(end_s * sr)]
REF_PATH = str(OUTPUT_DIR / "voice_ref.wav")
sf.write(REF_PATH, ref_audio, sr)

print(f"Voz de referencia: {REF_PATH}")
print(f"Duración: {end_s - start_s:.1f}s (de {start_s:.1f}s a {end_s:.1f}s del original)")

from IPython.display import Audio
Audio(REF_PATH)

## 9. Cargar XTTS-v2

Modelo multilingüe con clonación de voz cross-lingual. Primera carga descarga ~2GB.

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("XTTS-v2 listo.")

## 10. Generar audio por segmento

In [ ]:
from pathlib import Path

SEG_DIR = OUTPUT_DIR / "segments"
SEG_DIR.mkdir(exist_ok=True)

generated = []
for i, sub in enumerate(subs):
    text = sub.content.strip()
    if not text:
        generated.append(None)
        continue
    out_path = str(SEG_DIR / f"seg_{i:04d}.wav")
    try:
        tts.tts_to_file(
            text=text,
            file_path=out_path,
            speaker_wav=REF_PATH,
            language=TARGET_LANG,
            split_sentences=False,
        )
        generated.append(out_path)
    except Exception as e:
        print(f"  [WARN] seg {i+1} falló: {e}")
        generated.append(None)
    if (i + 1) % 10 == 0 or i == len(subs) - 1:
        print(f"[{i+1}/{len(subs)}] {text[:70]}")

print(f"\nTotal generados: {sum(1 for g in generated if g)}/{len(subs)}")

## 11. Ajustar velocidad de cada segmento (time-fit)

Si la traducción dura más que el slot original, comprime con rubberband (preserva el pitch). Si dura menos, deja silencio.

In [ ]:
import pyrubberband as pyrb
import soundfile as sf
import numpy as np

fitted = []
n_compressed = 0
n_capped = 0

for i, (sub, gen_path) in enumerate(zip(subs, generated)):
    if gen_path is None:
        fitted.append(None)
        continue
    target = (sub.end - sub.start).total_seconds()
    audio, sr = sf.read(gen_path)
    actual = len(audio) / sr

    if actual <= target * 1.02:
        # Encaja bien o sobra tiempo → dejamos como está
        fitted.append((audio, sr))
        continue

    speed = actual / target
    if speed > MAX_SPEED_RATIO:
        speed = MAX_SPEED_RATIO
        n_capped += 1
        print(f"  seg {i+1}: necesita {actual/target:.2f}x, limitado a {MAX_SPEED_RATIO}x (puede pisar el siguiente)")

    stretched = pyrb.time_stretch(audio, sr, speed)
    fitted.append((stretched, sr))
    n_compressed += 1

print(f"\nComprimidos: {n_compressed}, capeados al máximo: {n_capped}")

## 12. Construir audio final alineado al video

In [ ]:
# Duración real del video
probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", VIDEO_PATH
], capture_output=True, text=True)
video_duration = float(probe.stdout.strip())

sr_master = next(f[1] for f in fitted if f is not None)
master = np.zeros(int(video_duration * sr_master) + sr_master, dtype=np.float32)

for sub, fit in zip(subs, fitted):
    if fit is None:
        continue
    audio, _ = fit
    start_sample = int(sub.start.total_seconds() * sr_master)
    end_sample = min(start_sample + len(audio), len(master))
    master[start_sample:end_sample] += audio[:end_sample - start_sample].astype(np.float32)

# Normalizar para evitar clipping
peak = float(np.max(np.abs(master)))
if peak > 0.99:
    master = master * (0.99 / peak)

EN_AUDIO = str(OUTPUT_DIR / "en_audio.wav")
sf.write(EN_AUDIO, master, sr_master)
print(f"Audio final: {EN_AUDIO} ({video_duration:.1f}s)")

from IPython.display import Audio
Audio(EN_AUDIO)

## 13. Combinar con el video original

Reemplaza la pista de audio. El video queda intacto, solo cambia la voz.

In [ ]:
stem = Path(SRT_EN).stem.replace("-EN", "")
OUTPUT_VIDEO = str(OUTPUT_DIR / f"{stem}-{TARGET_LANG.upper()}.mp4")

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", VIDEO_PATH,
    "-i", EN_AUDIO,
    "-c:v", "copy",
    "-c:a", "aac", "-b:a", "192k",
    "-map", "0:v:0", "-map", "1:a:0",
    "-shortest",
    OUTPUT_VIDEO
], check=True)

print(f"✓ Listo: {OUTPUT_VIDEO}")
import os
print(f"  Tamaño: {os.path.getsize(OUTPUT_VIDEO) / 1024 / 1024:.1f} MB")

## 14. Descargar / guardar en Drive

In [ ]:
# Descarga directa al navegador
from google.colab import files
files.download(OUTPUT_VIDEO)

# O copiar a Drive (descomentá si lo montaste arriba)
# !cp "$OUTPUT_VIDEO" /content/drive/MyDrive/sta-ru-output/

---

## Modo batch (procesar los 82 videos)

Una vez que validaste que un video sale bien, usá `colab/batch_dub.py` para procesar todos en serie:

```python
!python colab/batch_dub.py --videos-dir /content/drive/MyDrive/sta-ru-videos --output-dir /content/drive/MyDrive/sta-ru-output
```

El script empareja cada video con su SRT por nombre y procesa todo de corrido.